# Credit Card Default Prediction — Training Notebook

**Stage 1.5** of the project (see `docs/progress.md`). Run this in Google Colab.

Dataset: [Default of Credit Card Clients](https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset)

Workflow reminder (full detail in `docs/mlflow-workflow.md`):
1. Everything here logs to a **local** MLflow tracking URI (`file:./mlruns`) — no network setup needed from Colab.
2. At the end of the session, zip `mlruns/` and download it, along with the best model's artifacts.
3. Drop `mlruns/` into `infra/mlflow/data/mlruns` on your machine to browse everything in the local MLflow UI.
4. Drop the exported model + preprocessing artifacts into `ml/artifacts/` for the FastAPI backend (Stage 2).

## 0. Setup

In [ ]:
!pip install -q "mlflow==2.16.2" torch scikit-learn imbalanced-learn pandas matplotlib seaborn optuna

# Pinned to 2.16.2 on purpose — matches infra/mlflow/Dockerfile exactly.
#
# Why this matters: `pip install mlflow` unpinned pulls MLflow 3.x, which put the
# FileStore backend (the plain-folder `mlruns/`, no database) into "maintenance mode" —
# `mlflow.set_experiment(...)` raises MlflowException unless you opt back in with
# MLFLOW_ALLOW_FILE_STORE=true. Setting that env var would silence the error, but it
# doesn't guarantee the on-disk format 3.x writes is byte-for-byte what 2.16.2 (running
# in the Docker server) reads back. Pinning both sides to the same version removes that
# risk entirely — this is the fix if you hit:
#   MlflowException: The filesystem tracking backend (e.g., './mlruns') is in
#   maintenance mode ...

import mlflow

# Local file-store tracking — matches the FileStore backend used by the
# Dockerized MLflow server, so mlruns/ can be copied over directly later.
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("credit-card-default")

## 1. Load Data

Download the dataset from Kaggle (via `kagglehub` or manual upload) into `ml/data/` conventions — in Colab, just load it into a DataFrame directly.

In [ ]:
# --- Option A (default): manual upload ---------------------------------------
# Download the CSV once from Kaggle (button on the dataset page), then run this
# cell and pick the file when the widget appears. No Kaggle account/API token
# juggling inside Colab — one less thing to debug on day one.
from google.colab import files

uploaded = files.upload()  # pick the downloaded CSV in the dialog
csv_filename = next(iter(uploaded))  # grabs whatever filename you uploaded

# --- Option B: kagglehub (uncomment if you'd rather not re-upload each session) ---
# import kagglehub
# path = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")
# csv_filename = f"{path}/UCI_Credit_Card.csv"

import pandas as pd

df = pd.read_csv(csv_filename)

# Quirk #1: the ID column is a row identifier, not a feature — drop it.
df = df.drop(columns=["ID"])

# Quirk #2 (easy to miss): the repayment-status columns are named
# PAY_0, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6 — there is no PAY_1. That's not a
# typo in this notebook, it's how the original dataset is labeled. PAY_0 is
# the most recent month, PAY_6 the oldest.

print(df.shape)
df.head()

## 2. Exploratory Data Analysis

- Target distribution (`default.payment.next.month`) — quantify the class imbalance
- Univariate distributions: `LIMIT_BAL`, `AGE`, `BILL_AMT1-6`, `PAY_AMT1-6`
- `PAY_0..PAY_6` repayment status patterns vs. default
- Data quality: undocumented codes in `EDUCATION` (0, 5, 6) and `MARRIAGE` (0)
- Correlation heatmap

In [ ]:
# 2.1 Target distribution — how imbalanced are we actually dealing with?
target = "default.payment.next.month"

counts = df[target].value_counts().sort_index()
pct = df[target].value_counts(normalize=True).sort_index() * 100

print("Class counts:\n", counts)
print("\nClass %:\n", pct.round(2))

# The number that matters most for everything downstream: if you did nothing
# clever and just predicted "no default" for every single customer, this is
# the accuracy you'd get for free — with zero predictive power.
naive_accuracy = counts[0] / counts.sum()
print(f"\nNaive 'always predict no-default' accuracy: {naive_accuracy:.2%}")
print("Any model has to beat THIS meaningfully on precision/recall — not just accuracy.")

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=["No Default (0)", "Default (1)"], y=counts.values, ax=ax, palette=["#4C72B0", "#C44E52"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f"{v}\n({pct.values[i]:.1f}%)", ha="center")
ax.set_ylabel("Number of customers")
ax.set_title("Target distribution — default.payment.next.month")
plt.show()

**Implication:** ~78/22 split — not extreme (like fraud's 99.9/0.1), but enough that accuracy is a
misleading headline metric. A model that just memorizes the majority class scores ~78% "accuracy" while
being useless. This is *why* Section 4 (Handling Class Imbalance) and the metrics we track from here on
(precision, recall, F1, PR-AUC) exist — keep this naive-accuracy number around as the floor everything else
has to clear.

In [ ]:
# 2.2 Missing values & duplicates — quick hygiene check before trusting anything else
print("Missing values per column (top 5, should be all 0 for this dataset):")
print(df.isnull().sum().sort_values(ascending=False).head())

print(f"\nExact duplicate rows: {df.duplicated().sum()}")

# Not glamorous, but skipping this step and finding out three sections later
# that you had NaNs silently propagating through a scaler is a worse afternoon.

### 2.3 Data quality: undocumented category codes

The dataset's documentation defines:
- `EDUCATION`: 1=graduate school, 2=university, 3=high school, 4=others
- `MARRIAGE`: 1=married, 2=single, 3=others

But real values in the columns don't stop there — let's look.

In [ ]:
print("EDUCATION value counts:")
print(df["EDUCATION"].value_counts().sort_index())
# -> you'll see 0, 5, 6 show up too — undocumented codes, almost certainly
#    data-entry artifacts or an "unknown" bucket that never made it into the docs.

print("\nMARRIAGE value counts:")
print(df["MARRIAGE"].value_counts().sort_index())
# -> 0 shows up here too, outside the documented 1/2/3.

# Not fixing these here — this is EDA, we're only *finding* the issue. The fix
# (folding 0/5/6 into EDUCATION's "others"=4, and 0 into MARRIAGE's "others"=3)
# happens in Section 3, where every preprocessing decision belongs together.

### 2.4 Numeric distributions — shape matters for a neural net

MLPs are sensitive to feature scale (that's *why* Section 3 ends in `StandardScaler`), and gradient
descent behaves badly when a feature is heavily right-skewed with a long tail of outliers — a few whales
with huge bill amounts can dominate the loss early in training. Let's see which columns actually look like
that before deciding anything.

In [ ]:
# LIMIT_BAL (credit limit) and AGE — the two "plain" continuous features
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["LIMIT_BAL"], bins=40, ax=axes[0], color="#4C72B0")
axes[0].set_title("LIMIT_BAL distribution")
sns.histplot(df["AGE"], bins=40, ax=axes[1], color="#55A868")
axes[1].set_title("AGE distribution")
plt.tight_layout()
plt.show()

print("LIMIT_BAL skew:", df["LIMIT_BAL"].skew().round(2))
print("AGE skew:", df["AGE"].skew().round(2))

# BILL_AMT1 / PAY_AMT1 — representative of the 6 bill/payment columns each.
# Plotting raw vs log1p side-by-side to make the skew concrete rather than
# just quoting a skew number.
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
sns.histplot(df["BILL_AMT1"], bins=50, ax=axes[0, 0], color="#C44E52")
axes[0, 0].set_title(f"BILL_AMT1 (raw) — skew={df['BILL_AMT1'].skew():.2f}")
sns.histplot(np.sign(df["BILL_AMT1"]) * np.log1p(np.abs(df["BILL_AMT1"])), bins=50, ax=axes[0, 1], color="#C44E52")
axes[0, 1].set_title("BILL_AMT1 (signed log1p) — for comparison only")

sns.histplot(df["PAY_AMT1"], bins=50, ax=axes[1, 0], color="#8172B2")
axes[1, 0].set_title(f"PAY_AMT1 (raw) — skew={df['PAY_AMT1'].skew():.2f}")
sns.histplot(np.log1p(df["PAY_AMT1"]), bins=50, ax=axes[1, 1], color="#8172B2")
axes[1, 1].set_title("PAY_AMT1 (log1p) — for comparison only")
plt.tight_layout()
plt.show()

# Note: BILL_AMT can be negative (overpayment/credit balance), which is why the
# signed-log trick is used instead of a plain log1p — plain log1p breaks on
# negative input. This plot is just showing you the shape difference; whether
# we actually apply a log transform is a Section 3 decision, made together
# with the engineered ratio features (a ratio feature can absorb some of this
# skew on its own, which may make an explicit log transform unnecessary).

### 2.5 Repayment status (`PAY_0..PAY_6`) vs. default — the headline signal

`PAY_0` is last month's repayment status: -1/0 roughly mean "paid on time / revolving credit used
properly", and 1, 2, 3... mean "N months late". If there's one relationship in this dataset that should be
strong, it's this one — let's check, and use it as the sanity check for everything else: if a supposedly
"engineered" feature in Section 3 correlates with default *less* than raw `PAY_0`, that's a signal the
engineering didn't add much.

In [ ]:
pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, pay_cols):
    default_rate_by_status = df.groupby(col)[target].mean().sort_index()
    n_by_status = df.groupby(col).size().sort_index()
    sns.barplot(x=default_rate_by_status.index, y=default_rate_by_status.values, ax=ax, color="#C44E52")
    ax.axhline(naive_accuracy_complement := df[target].mean(), color="gray", linestyle="--", linewidth=1)
    ax.set_title(col)
    ax.set_ylabel("Default rate")
    ax.set_xlabel("Repayment status code")
plt.suptitle("Default rate by repayment status, per month (dashed line = overall default rate)", y=1.02)
plt.tight_layout()
plt.show()

# Reading this: for PAY_0, status <= 0 (paid on time) sits near/below the dashed
# overall-rate line; each step up in "months late" pushes the default rate up,
# often steeply. That monotonic climb is exactly the kind of pattern a linear
# model AND a neural net can both exploit easily — this is your strongest
# individual predictor before any feature engineering happens at all.

### 2.6 Correlation heatmap — which raw features already "know" the target?

A heatmap won't catch non-linear relationships (a neural net can), but it's a fast way to see which raw
columns already carry signal and which look like noise before you've engineered anything. Worth revisiting
after Section 3 — if an engineered feature doesn't beat its raw ingredients here, question whether it earns
its place in the model.

In [ ]:
corr = df.corr(numeric_only=True)

# Full heatmap is dense (23 features) — also print just the target correlations,
# sorted, since that ranked list is usually more actionable than eyeballing a grid.
target_corr = corr[target].drop(target).sort_values(key=abs, ascending=False)
print("Features ranked by |correlation| with default:")
print(target_corr.round(3))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, ax=ax)
ax.set_title("Correlation matrix — all numeric features")
plt.tight_layout()
plt.show()

# Expect PAY_0..PAY_6 to dominate the top of that ranked list (confirms 2.5),
# LIMIT_BAL to show a modest negative correlation (higher credit limit ~ lower
# default, likely because it's a proxy for creditworthiness the bank already
# assessed), and the six BILL_AMT columns to correlate strongly WITH EACH
# OTHER (multicollinearity — a customer's bill in month 1 is similar to month
# 2) more than with the target individually. That last point is itself useful:
# six highly-correlated raw columns is a candidate for feature engineering
# (e.g. a trend/slope feature) instead of feeding all six in raw.

### 2.7 Default rate across demographic slices

Not because these will necessarily be strong predictors (they usually aren't, compared to `PAY_0`), but
because it's worth knowing whether the model's errors will skew across sex/education/marital-status groups
before it's deployed — that's an error-analysis and fairness question worth having eyes on early, not
something to discover after Section 9.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sex_labels = {1: "Male", 2: "Female"}
sns.barplot(x=df["SEX"].map(sex_labels), y=df[target], ax=axes[0], color="#4C72B0", errorbar=None)
axes[0].set_title("Default rate by SEX")
axes[0].set_ylabel("Default rate")

sns.barplot(x=df["EDUCATION"], y=df[target], ax=axes[1], color="#55A868", errorbar=None)
axes[1].set_title("Default rate by EDUCATION (raw codes incl. undocumented)")

sns.barplot(x=df["MARRIAGE"], y=df[target], ax=axes[2], color="#8172B2", errorbar=None)
axes[2].set_title("Default rate by MARRIAGE (raw codes incl. undocumented)")

plt.tight_layout()
plt.show()

# AGE as a binned view — raw age is noisy, bucketing makes any trend visible
age_bins = pd.cut(df["AGE"], bins=[20, 30, 40, 50, 60, 80], labels=["21-30", "31-40", "41-50", "51-60", "60+"])
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=age_bins, y=df[target], ax=ax, color="#C44E52", errorbar=None)
ax.set_title("Default rate by age bracket")
ax.set_ylabel("Default rate")
plt.show()

### 2.8 EDA summary → what it decides for Section 3

| Finding | Decision it drives |
|---|---|
| ~78/22 class split | Track precision/recall/F1/PR-AUC everywhere, not accuracy; revisit imbalance handling in Section 4 |
| `EDUCATION`/`MARRIAGE` have undocumented codes (0, 5, 6 / 0) | Consolidate into the documented "others" bucket before encoding |
| `BILL_AMT*`/`PAY_AMT*` are heavily right-skewed | `StandardScaler` alone may not be enough — consider engineered ratio/trend features that are naturally less skewed than the raw amounts |
| `PAY_0..PAY_6` dominate correlation with target, and climb monotonically with lateness | Strongest raw signal — any engineered feature (delinquency streak, etc.) should be judged against beating this baseline, not replacing it |
| `BILL_AMT1..6` are highly collinear with each other | A trend/slope feature across the 6 months may capture more than 6 raw correlated columns |
| Demographic slices (`SEX`/`EDUCATION`/`MARRIAGE`/`AGE`) show smaller, noisier effects than `PAY_0` | Still worth encoding as features, but keep an eye on them in Section 9's error analysis for skewed error rates |

Nothing gets fixed in this section on purpose — EDA's job is to *decide* what Section 3 needs to do and
*why*, not to do it. Next: Section 3, acting on this table.

## 3. Preprocessing & Feature Engineering

- Consolidate undocumented `EDUCATION`/`MARRIAGE` categories
- Encode categoricals (`SEX`, `EDUCATION`, `MARRIAGE`)
- Engineered features: payment-to-bill ratios, delinquency streak length across `PAY_0..PAY_6`, spending trend slope across `BILL_AMT1..6`
- Scale numeric features (fit on train only — `StandardScaler`)
- Stratified train / validation / test split (stratify on target, given the imbalance)
- Persist the fitted scaler + feature column order — needed later for backend inference

In [ ]:
# TODO: preprocessing + feature engineering

## 4. Handling Class Imbalance

Compare, as separate logged MLflow runs:
- Baseline (no handling)
- Class-weighted loss (`pos_weight` in `BCEWithLogitsLoss`)
- SMOTE oversampling (train split only)
- Random undersampling (train split only)

Evaluate on precision / recall / F1 / PR-AUC — **not** accuracy alone.

In [ ]:
# TODO: imbalance strategy comparison

## 5. MLP Model (PyTorch)

Define the architecture once, parametrized (hidden layer sizes, dropout rate) so it can be reused across the
hyperparameter search below. Keep `model_config.json`-style parameters explicit — the backend will need to
reconstruct this exact architecture at inference time.

In [ ]:
# TODO: nn.Module definition

## 6. Weight Initialization Experiments

Compare zero init (baseline, expect it to fail/stagnate), Xavier/Glorot, and He initialization. Log each as a run.

In [ ]:
# TODO: weight init comparison

## 7. Regularization Experiments

Dropout, Batch Normalization, L2 weight decay, Early Stopping — ablate individually and combined, log each combination.

In [ ]:
# TODO: regularization ablations

## 8. Hyperparameter Tuning

- Grid search and random search over: learning rate, batch size, hidden layer sizes, dropout rate, optimizer (SGD / Momentum / RMSProp / Adam / AdamW)
- Optional stretch: Optuna study for a more efficient search
- Log every trial as an MLflow run (params + metrics), so the comparison is queryable later

In [ ]:
# TODO: hyperparameter search

## 9. Final Model Evaluation & Error Analysis

- Select best run from MLflow (`mlflow.search_runs`, sort by chosen metric — decide precision vs. recall priority first)
- Confusion matrix, classification report, ROC and PR curves on the held-out test set
- Inspect misclassified examples — any pattern (e.g. borderline `PAY_0` values)?

In [ ]:
# TODO: final evaluation

## 10. Export Artifacts

Save, then download and place into `ml/artifacts/`:
- `model/model.pt` — trained weights
- `model/model_config.json` — architecture params needed to reconstruct the `nn.Module`
- `preprocessing/scaler.pkl` — fitted scaler
- `preprocessing/feature_columns.json` — exact column order/names expected at inference
- `metrics/evaluation_report.json` — final metrics for the README/docs

In [ ]:
# TODO: export artifacts + zip mlruns/ for download